# Training the selected legacy MLP

The notebook trains only the MLP selected in the legacy analysis. Its
architecture is unchanged: `MLPClassifier(hidden_layer_sizes=(64, 32),
activation="relu", alpha=1e-4, learning_rate_init=1e-3, max_iter=200,
random_state=42)` preceded by a constant-zero imputer.

The three retained input columns are `mean_o`, `std_o`, and `skew_o`.
The timer covers only `pipeline.fit(X_train, y_train)`.


In [ ]:
from __future__ import annotations

import json
import platform
import subprocess
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd


WORK_ROOT = Path.cwd().resolve()

DATA_FOLDER = Path("/hercules/results/akazantsev/rfim_dataset")
META_PATH = DATA_FOLDER / "B0531+21_59000_48386_subset_channels_meta.csv"
SPLIT_PATH = DATA_FOLDER / "split_indices.npz"
PROFILES_PATH = DATA_FOLDER / "B0531+21_59000_48386_subset_channels.npy"

# The training subset deliberately retains only statistical features and labels.
# These full files retain channel and segment identity and are used only by the
# inference-timing notebook, where one input must correspond to a real 256-channel
# observation segment.
FULL_META_PATH = DATA_FOLDER / "B0531+21_59000_48386_channels_meta.csv"
FULL_PROFILES_PATH = DATA_FOLDER / "B0531+21_59000_48386_channels.npy"
SUBSET_SOURCE_INDICES_PATH = DATA_FOLDER / "B0531+21_59000_48386_subset_indices.npy"

# Change the tag only for a deliberate new experiment. Existing results are never overwritten.
RUN_TAG = "b0531_legacy_performance_v1"
RUN_ROOT = WORK_ROOT / "outputs" / "performance_comparison" / RUN_TAG


def json_ready(value):
    if isinstance(value, dict):
        return {key: json_ready(item) for key, item in value.items()}
    if isinstance(value, (np.integer, np.floating)):
        return value.item()
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, (list, tuple)):
        return [json_ready(item) for item in value]
    return value


def write_json(path: Path, payload: dict) -> None:
    with path.open("w", encoding="utf-8") as handle:
        json.dump(json_ready(payload), handle, indent=2, sort_keys=True)
        handle.write("\n")


def git_revision() -> str:
    try:
        return subprocess.check_output(
            ["git", "rev-parse", "HEAD"],
            cwd=WORK_ROOT,
            text=True,
            stderr=subprocess.DEVNULL,
        ).strip()
    except (FileNotFoundError, subprocess.CalledProcessError):
        return None


In [ ]:
import joblib
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    confusion_matrix,
    log_loss,
    precision_recall_fscore_support,
    roc_auc_score,
)
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline

def best_threshold_by_f1(y_true, scores):
    best_threshold, best_f1 = 0.5, -1.0
    for threshold in np.linspace(0.01, 0.99, 99):
        prediction = (scores >= threshold).astype(int)
        _, _, f1, _ = precision_recall_fscore_support(
            y_true, prediction, average="binary", zero_division=0
        )
        if f1 > best_f1:
            best_threshold, best_f1 = float(threshold), float(f1)
    return best_threshold, best_f1

def eval_binary(y_true, scores, threshold):
    prediction = (scores >= threshold).astype(int)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, prediction, average="binary", zero_division=0
    )
    tn, fp, fn, tp = confusion_matrix(y_true, prediction, labels=[0, 1]).ravel()
    both_classes = len(np.unique(y_true)) == 2
    return {
        "TN": int(tn), "FP": int(fp), "FN": int(fn), "TP": int(tp),
        "threshold": float(threshold),
        "accuracy": float(accuracy_score(y_true, prediction)),
        "precision": float(precision), "recall": float(recall), "f1": float(f1),
        "roc_auc": float(roc_auc_score(y_true, scores)) if both_classes else None,
        "pr_auc": float(average_precision_score(y_true, scores)) if both_classes else None,
        "logloss": float(log_loss(y_true, np.c_[1 - scores, scores], labels=[0, 1])) if both_classes else None,
    }

RANDOM_STATE = 42
SELECTED_FEATURES = ["mean_o", "std_o", "skew_o"]
output_dir = RUN_ROOT / "mlp_orig_top3"
if output_dir.exists():
    raise FileExistsError(
        f"{output_dir} already exists. Choose a new RUN_TAG rather than overwrite it."
    )
output_dir.mkdir(parents=True)


In [ ]:
# Preserve the legacy metadata and split-loading contract.
meta = pd.read_csv(META_PATH)
meta["label"] = meta["label"].fillna("None")
splits = np.load(SPLIT_PATH)

train_idx = np.asarray(splits["train_idx"], dtype=int)
val_idx = np.asarray(splits["val_idx"], dtype=int)
test_idx = np.asarray(splits["test_idx"], dtype=int)

missing_columns = set(SELECTED_FEATURES).difference(meta.columns)
if missing_columns:
    raise ValueError(f"Metadata is missing selected features: {sorted(missing_columns)}")

x_all = meta[SELECTED_FEATURES].apply(pd.to_numeric, errors="coerce")
y_all = meta["label"].eq("NBRFI").to_numpy(dtype=int)

x_train, x_validation, x_test = (
    x_all.iloc[indices] for indices in (train_idx, val_idx, test_idx)
)
y_train, y_validation, y_test = (
    y_all[indices] for indices in (train_idx, val_idx, test_idx)
)

print("Selected features:", SELECTED_FEATURES)
print("Train / validation / test:", len(train_idx), len(val_idx), len(test_idx))


In [ ]:
# This is the exact final MLP pipeline from the legacy selected-top-3 run.
pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="constant", fill_value=0.0)),
    ("clf", MLPClassifier(
        hidden_layer_sizes=(64, 32),
        activation="relu",
        alpha=1e-4,
        learning_rate_init=1e-3,
        max_iter=200,
        random_state=RANDOM_STATE,
    )),
])

training_started = time.perf_counter()
pipeline.fit(x_train, y_train)
training_wall_clock_s = time.perf_counter() - training_started

validation_scores = pipeline.predict_proba(x_validation)[:, 1]
threshold, validation_f1 = best_threshold_by_f1(y_validation, validation_scores)
test_scores = pipeline.predict_proba(x_test)[:, 1]
test_metrics = eval_binary(y_test, test_scores, threshold=threshold)

print(f"Training wall-clock time: {training_wall_clock_s:.3f} s")
print(f"Validation-selected threshold: {threshold:.3f}")
print(f"Validation F1: {validation_f1:.4f}")
print(f"Test F1: {test_metrics['f1']:.4f}")


In [ ]:
bundle_path = output_dir / "mlp_orig_top3.joblib"
joblib.dump(
    {
        "model_name": "MLP_orig_top3",
        "pipeline": pipeline,
        "feature_cols": SELECTED_FEATURES,
        "threshold": float(threshold),
        "validation_f1_at_threshold": float(validation_f1),
    },
    bundle_path,
)

summary = {
    "run_tag": RUN_TAG,
    "model": "MLPClassifier",
    "pipeline": "SimpleImputer(constant=0.0) -> MLPClassifier(64, 32, relu)",
    "selected_features": SELECTED_FEATURES,
    "code_revision": git_revision(),
    "python_version": platform.python_version(),
    "dataset": {
        "metadata_path": META_PATH,
        "split_path": SPLIT_PATH,
        "n_train": len(train_idx),
        "n_validation": len(val_idx),
        "n_test": len(test_idx),
    },
    "training_protocol": {
        "max_iter": 200,
        "hidden_layer_sizes": [64, 32],
        "activation": "relu",
        "alpha": 1e-4,
        "learning_rate_init": 1e-3,
        "random_state": RANDOM_STATE,
        "timer_scope": "pipeline.fit(X_train, y_train) only; excludes CSV loading and validation/test scoring",
    },
    "training_wall_clock_s": training_wall_clock_s,
    "n_iter": int(pipeline.named_steps["clf"].n_iter_),
    "threshold": threshold,
    "validation_f1": validation_f1,
    "test_metrics": test_metrics,
    "artifacts": {"model_bundle": bundle_path},
}
write_json(output_dir / "training_summary.json", summary)

print(json.dumps(json_ready(summary), indent=2))
